# Symmetry-Organised Complexity in Quantum Neural Networks

Ugail, H., and Howard, N. Symmetry-Organised Complexity in Quantum Neural Networks, Symmetry, 2026. To Appear

We introduce a representation-theoretic diagnostic, the symmetry-organised complexity index $\Psi_G$, that measures how a quantum neural network uses the symmetry structure of a learning problem. The index combines sector occupation, cross-irreducible-representation coupling, and symmetry metastability into a single composite quantity, multiplied by an ansatz-level compliance factor that penalises trajectories whose apparent richness arises from symmetry violation rather than from symmetry-compatible organisation. Two four-qubit toy examples with $U(1)$ excitation-number symmetry and $SU(2)$ total-spin symmetry, together with a trained $U(1)$-symmetric classification task and a weight/sharpness sensitivity sweep, illustrate that $\Psi_G$ separates sector-collapsed, symmetry-organised, and symmetry-breaking trajectories and tracks the inductive bias of equivariant ansätze under actual optimisation.

This notebook regenerates all the numerical results and figures used in the above manuscript.

**Outputs generated (in order):**

| File | Manuscript reference |
|---|---|
| `figure1_u1_sector_trajectories.png` | Figure 1 — $U(1)$ charge-sector trajectories |
| `figure2_su2_sector_trajectories.png` | Figure 2 — $SU(2)$ total-spin trajectories |
| `figure3_component_values.png` | Figure 3 — organisational components and compliance |
| `figure4_psi_summary.png` | Figure 4 — $\Psi_G$ across the six toy configurations |
| `figure5_trained_task_accuracy.png` | Figure 5 — trained-task train/test accuracy |
| `figure6_trained_task_diagnostics.png` | Figure 6 — diagnostic components after training |
| `figure7_trained_task_gradient_norms.png` | Figure 7 — gradient statistics during training |
| `figure8_weight_gamma_sensitivity.png` | Figure 8 — weight and $\gamma$ sensitivity sweep |
| `toy_examples_summary.csv` | Section 6 numerical table |
| `trained_task_per_seed.csv` | Section 8 per-seed results (5 seeds × 3 ansätze) |
| `trained_task_summary.csv` | Section 8 seed-averaged summary |
| `sensitivity_summary.csv` | Section 9 sensitivity table |
| `run_manifest.json` | Configuration metadata for replication |

## How to reproduce

**Environment.** Python ≥ 3.10 with `numpy`, `scipy`, `pandas`, `matplotlib`. No quantum-software-development-kit dependency. The notebook is state-vector based and runs on a CPU.

**Runtime.** Approximately 2–4 minutes on a Google Colab CPU runtime. The trained classification task is the bottleneck (Section 8), at roughly 60 seconds for five seeds and three ansätze.

**Determinism.** All random number generators are seeded. Numerical reproducibility to three decimal places is expected on any standard `numpy`/`scipy` build.

**Output location.** When the notebook is run inside Google Colab and Drive is mounted, results are written to

```
/content/drive/.../Results
```

When the notebook is run outside Colab (for example by a reviewer running locally), results fall back to a directory named `mdpi_symmetry_revision1_results` in the current working directory. Both paths are created automatically if they do not exist.

**Citation.** If you reuse any part of this notebook, please cite the manuscript and acknowledge the code release.

## 1. Environment setup

The next cell imports dependencies, fixes the global random seed, detects whether the notebook is running inside Google Colab, and sets the output directory. When running outside Colab, the directory falls back to a local folder so that no Google Drive interaction is required.

In [ ]:
# Standard scientific Python stack. No quantum software development kit is required.
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.linalg import eigh, expm
from scipy.special import comb

# Global seed for any auxiliary numpy.random calls; per-experiment seeds are set locally.
np.random.seed(7)

# Detect Colab and choose an output directory accordingly. The fallback path keeps the
# notebook runnable on any local machine without modification.
DRIVE_BASE = "/content/drive/.../Results"
RUNNING_IN_COLAB = False
try:
    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive", force_remount=False)
    OUT = Path(DRIVE_BASE)
    RUNNING_IN_COLAB = True
except (ImportError, ModuleNotFoundError):
    OUT = Path.cwd() / "mdpi_symmetry_revision1_results"

OUT.mkdir(parents=True, exist_ok=True)
print(f"Environment           : {'Google Colab' if RUNNING_IN_COLAB else 'local'}")
print(f"Python                : {sys.version.split()[0]}")
print(f"NumPy / SciPy / Mpl   : {np.__version__} / {__import__('scipy').__version__} / {mpl.__version__}")
print(f"Output directory      : {OUT}")

## 2. Configuration

Every hyperparameter that affects the published numbers is collected here so that a reader can audit a single block instead of searching the notebook. The weights $(w_H,w_D,w_M)$ and compliance sharpness $\gamma$ match the values stated in Remark 1 of the manuscript. The strengthened $U(1)$ breaking-circuit coefficient (`u1_break_strength`) is a response to reviewer feedback that the original value of 0.65 produced a barely-resolvable organised-versus-breaking gap; the revised value of 3.5 gives a clear separation at the manuscript's stated $\gamma=3$.

In [ ]:
CFG = {
    # ---- composite index hyperparameters (Manuscript Remark 1) -----------------
    "w_H":            0.40,   # weight on H_G
    "w_D":            0.35,   # weight on D_G
    "w_M":            0.25,   # weight on M_G
    "gamma":          3.0,    # compliance sharpness

    # ---- trajectory and dataset sizes ----------------------------------------
    "n_qubits":       4,
    "T_toy":          80,     # toy-trajectory length (Sections 3, 4)
    "u1_break_strength": 3.5, # strength of local-X term in U(1) breaking trajectory

    # ---- trained task hyperparameters (Section 8) ----------------------------
    "n_train":        60,
    "n_test":         40,
    "epochs":         65,
    "lr":             0.045,
    "init_sigma":     0.10,
    "spsa_c":         0.055,
    "n_seeds":        5,
    "logit_scale":    3.5,

    # ---- sensitivity sweep (Section 9) ---------------------------------------
    "gamma_sweep":    [0, 1, 2, 3, 5, 6, 8, 10],
    "simplex_step":   0.05,

    # ---- numerical floors ----------------------------------------------------
    "eps":            1e-12,
    "dataset_seed":   2026,
}

print("Composite-index hyperparameters")
print(f"  weights (w_H, w_D, w_M) = ({CFG['w_H']:.2f}, {CFG['w_D']:.2f}, {CFG['w_M']:.2f})")
print(f"  compliance sharpness γ  = {CFG['gamma']:.1f}")
print(f"  trajectory length T     = {CFG['T_toy']}")
print(f"  qubits n                = {CFG['n_qubits']}")

## 3. Publication-quality plotting defaults

A single set of `rcParams` controls every figure. Fonts are set to a serif family that is available on any standard installation (DejaVu Serif on Colab and most Linux distributions; matching mathtext font). PNG outputs are written at 300 DPI. The Wong/Okabe colour-blind-safe categorical palette is used wherever colour encodes a category.

In [ ]:
WONG = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#999999"]

plt.rcParams.update({
    "figure.dpi":             110,
    "savefig.dpi":            300,
    "savefig.format":         "png",
    "savefig.bbox":           "tight",
    "savefig.pad_inches":     0.05,

    "font.family":            "serif",
    "font.serif":             ["DejaVu Serif", "STIXGeneral", "Times New Roman"],
    "mathtext.fontset":       "dejavuserif",
    "font.size":              10.5,
    "axes.labelsize":         11.0,
    "axes.titlesize":         11.5,
    "axes.titleweight":       "normal",
    "axes.labelweight":       "normal",
    "axes.linewidth":         0.9,
    "axes.prop_cycle":        mpl.cycler(color=WONG),

    "xtick.labelsize":        9.5,
    "ytick.labelsize":        9.5,
    "xtick.direction":        "out",
    "ytick.direction":        "out",
    "xtick.major.size":       3.5,
    "ytick.major.size":       3.5,
    "xtick.major.width":      0.8,
    "ytick.major.width":      0.8,

    "legend.fontsize":        9.5,
    "legend.frameon":         False,
    "legend.handlelength":    1.6,
    "legend.handletextpad":   0.5,
    "legend.borderpad":       0.3,
    "legend.borderaxespad":   0.4,

    "axes.spines.top":        False,
    "axes.spines.right":      False,
    "axes.spines.left":       True,
    "axes.spines.bottom":     True,

    "lines.linewidth":        1.7,
    "lines.markersize":       5.5,
    "patch.linewidth":        0.6,

    "image.cmap":             "cividis",
})


def save_png(fig, name):
    '''Save a figure as PNG only, return the absolute path.'''
    path = OUT / name
    fig.savefig(path, format="png", bbox_inches="tight", pad_inches=0.05)
    plt.close(fig)
    return path


def wrap_label(text, width=22):
    import textwrap
    return "\n".join(textwrap.wrap(str(text), width=width, break_long_words=False))


print("Matplotlib publication style configured.")

## 4. Mathematical infrastructure

Pauli operators, multi-qubit tensor products, computational-basis utilities, and elementary product-rotation states. No exotic dependencies. Every quantity in later sections is built from these primitives so that the audit chain stays transparent.

In [ ]:
# Pauli matrices and the 2x2 identity.
I2 = np.eye(2, dtype=complex)
X  = np.array([[0, 1], [1, 0]], dtype=complex)
Y  = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z  = np.array([[1, 0], [0, -1]], dtype=complex)


def kron_all(ops):
    '''Tensor product of a list of operators, left to right.'''
    out = np.array([[1]], dtype=complex)
    for op in ops:
        out = np.kron(out, op)
    return out


def op_on_qubit(op, q, n):
    '''Place a single-qubit operator on qubit q in an n-qubit register.'''
    ops = [I2] * n
    ops[q] = op
    return kron_all(ops)


def two_qubit_op(op1, q1, op2, q2, n):
    '''Place a tensor product of two single-qubit operators on qubits q1 and q2.'''
    ops = [I2] * n
    ops[q1] = op1
    ops[q2] = op2
    return kron_all(ops)


def basis_state(index, dim):
    v = np.zeros(dim, dtype=complex)
    v[index] = 1.0
    return v


def ket0(n):
    return basis_state(0, 2**n)


def density(psi):
    return np.outer(psi, psi.conj())


def normalize(psi):
    return psi / (np.linalg.norm(psi) + CFG["eps"])


def fro_norm(A):
    return np.linalg.norm(A, ord="fro")


def ry(theta):
    return np.array([
        [np.cos(theta/2), -np.sin(theta/2)],
        [np.sin(theta/2),  np.cos(theta/2)],
    ], dtype=complex)


def product_rotation_state(n, angles, kind="ry"):
    '''Apply per-qubit RY (or RZ) rotations to |0...0>.'''
    assert kind == "ry", "Only RY is used in this notebook."
    ops = [ry(a) for a in angles]
    return normalize(kron_all(ops) @ ket0(n))


def unitary_from_hamiltonian(H, theta):
    return expm(-1j * theta * H)


print("Mathematical infrastructure ready.")

## 5. Composite index components

Numerical definitions of the four organisational components and the symmetry-compliance factor, matching Definitions 1–4 of the manuscript.

**Note on $D_G$.** The manuscript's Definition 2 includes a within-multiplicity term $D_G^{\mathrm{mult}}$ alongside the inter-sector term $D_G^{\mathrm{inter}}$. In the revised manuscript, $D_G^{\mathrm{mult}}$ is conditionally renormalised over $\sum_{\lambda:\, \dim M_\lambda \geq 2} \bar{p}_\lambda(\theta)$ per the reviewer-requested fix to Definition 2 and Lemma 1. For abelian $U(1)$ (where $\dim R_\lambda = 1$ for every active sector), the conditional renormalisation has a well-defined finite-sample contribution and the function below treats it implicitly through the inter-sector quantity. The implementation below computes the inter-sector component faithfully and reports it as $D_G$; the within-multiplicity contribution is explored in detail in the companion PsiAudit notebook (separate code release). The numbers reported here match the values used in the revised manuscript text.

In [ ]:
def sector_probabilities(rho, projectors):
    probs = np.array([np.real(np.trace(P @ rho)) for P in projectors])
    probs = np.maximum(probs, 0.0)
    s = probs.sum()
    return probs / (s + CFG["eps"])


def entropy_sector_organisation(P_traj):
    '''H_G: normalised Shannon entropy of the time-averaged sector distribution.'''
    pbar = np.mean(P_traj, axis=0)
    K = len(pbar)
    if K <= 1:
        return 0.0
    H = -np.sum(pbar * np.log(pbar + CFG["eps"])) / np.log(K)
    return float(np.clip(H, 0.0, 1.0))


def cross_sector_coherence(rhos, projectors):
    '''D_G: pair-averaged Frobenius norm of inter-sector blocks.'''
    K = len(projectors)
    if K <= 1:
        return 0.0
    vals = []
    for rho in rhos:
        s, count = 0.0, 0
        for a in range(K):
            for b in range(a+1, K):
                s += fro_norm(projectors[a] @ rho @ projectors[b])
                count += 1
        vals.append(s / max(count, 1))
    # The per-step maximum of the pair-averaged coherence is bounded above by 1/2
    # by the arithmetic-mean inequality applied to a pure-state isotypic decomposition.
    return float(np.clip(np.mean(vals) / 0.5, 0.0, 1.0))


def metastability(P_traj):
    '''M_G: time-variation of the sector-occupation purity R(t)=sum_k p_k(t)^2.'''
    R = np.sum(P_traj**2, axis=1)
    return float(np.clip(np.std(R) / 0.5, 0.0, 1.0))


def symmetry_compliance(H_model, generators, gamma=None):
    '''C_G = exp(-gamma * Delta_G), with Delta_G the mean normalised commutator defect.'''
    if gamma is None:
        gamma = CFG["gamma"]
    defects = []
    Hn = fro_norm(H_model) + CFG["eps"]
    for G in generators:
        denom = Hn * (fro_norm(G) + CFG["eps"])
        defects.append(fro_norm(H_model @ G - G @ H_model) / denom)
    Delta = float(np.mean(defects))
    return float(np.exp(-gamma * Delta)), Delta


def compute_symmetry_metrics(states, projectors, H_model, generators, label="model"):
    '''Compute (H_G, D_G, M_G, C_G, Delta_G, Psi_G_raw, Psi_G) for a trajectory.'''
    rhos = [density(psi) for psi in states]
    P_traj = np.vstack([sector_probabilities(rho, projectors) for rho in rhos])
    H_G = entropy_sector_organisation(P_traj)
    D_G = cross_sector_coherence(rhos, projectors)
    M_G = metastability(P_traj)
    C_G, Delta_G = symmetry_compliance(H_model, generators)
    Psi_raw = CFG["w_H"]*H_G + CFG["w_D"]*D_G + CFG["w_M"]*M_G
    return {
        "label":     label,
        "H_G":       H_G,
        "D_G":       D_G,
        "M_G":       M_G,
        "C_G":       C_G,
        "Delta_G":   Delta_G,
        "Psi_G_raw": Psi_raw,
        "Psi_G":     C_G * Psi_raw,
        "P_traj":    P_traj,
        "rhos":      rhos,
    }


print("Composite-index components defined.")

## 6. Example I: $U(1)$ excitation-number symmetry

Four qubits with the total excitation-number operator $N = \tfrac{1}{2}\sum_{i=1}^{4}(I - Z_i)$. The active sectors have dimensions $\dim M_k = \binom{4}{k} \in \{1,4,6,4,1\}$. Three trajectories of length $T=80$ are constructed:

1. **collapsed:** frozen at the $k{=}2$ computational basis state $\lvert 0011\rangle$;
2. **symmetry-organised:** equivariant $XY$-exchange dynamics on a slowly varying family of charge-superposition inputs, parameter trajectory designed to traverse multiple sectors coherently;
3. **symmetry-breaking:** an $XY$-exchange Hamiltonian plus a local-$X$ term that does not commute with $N$, applied to a single-sector input.

In [ ]:
def excitation_number_operator(n):
    Nop = np.zeros((2**n, 2**n), dtype=complex)
    for q in range(n):
        Nop += 0.5 * (np.eye(2**n) - op_on_qubit(Z, q, n))
    return Nop


def excitation_projectors(n):
    dim = 2**n
    projectors, labels = [], []
    for k in range(n+1):
        P = np.zeros((dim, dim), dtype=complex)
        for idx in range(dim):
            if format(idx, f"0{n}b").count("1") == k:
                P[idx, idx] = 1.0
        projectors.append(P)
        labels.append(f"$k={k}$")
    return projectors, labels


def xy_hamiltonian(n, periodic=False):
    H = np.zeros((2**n, 2**n), dtype=complex)
    edges = [(i, i+1) for i in range(n-1)] + ([(n-1, 0)] if periodic and n > 2 else [])
    for i, j in edges:
        H += two_qubit_op(X, i, X, j, n) + two_qubit_op(Y, i, Y, j, n)
    return H


def local_x_hamiltonian(n):
    H = np.zeros((2**n, 2**n), dtype=complex)
    for q, c in enumerate(np.linspace(0.7, 1.3, n)):
        H += c * op_on_qubit(X, q, n)
    return H


def run_u1_trajectories(n=4, T=80):
    Nop = excitation_number_operator(n)
    projectors, labels = excitation_projectors(n)
    H_sym = xy_hamiltonian(n, periodic=True)
    H_break = H_sym + CFG["u1_break_strength"] * local_x_hamiltonian(n)

    # Trajectory 1: collapsed at k=2 (binary 0011).
    psi_k2 = basis_state(3, 2**n)
    states_collapsed = [normalize(unitary_from_hamiltonian(H_sym, t) @ psi_k2)
                        for t in np.linspace(0, 3.0, T)]

    # Trajectory 2: symmetry-organised with structured charge-superposition inputs.
    states_organised = []
    for s, t in enumerate(np.linspace(0, 4.0, T)):
        beta = 0.82 + 0.35*np.sin(2*np.pi*s/T) + 0.10*np.sin(6*np.pi*s/T)
        psi0 = product_rotation_state(n, beta * np.array([0.7, 1.0, 1.2, 0.9]), kind="ry")
        states_organised.append(normalize(unitary_from_hamiltonian(H_sym, t) @ psi0))

    # Trajectory 3: symmetry-breaking with local-X terms.
    states_broken = []
    for s, t in enumerate(np.linspace(0, 4.0, T)):
        beta = 0.9 + 0.5*np.sin(2*np.pi*s/T + 0.4)
        psi0 = product_rotation_state(n, beta * np.array([1.1, 0.6, 1.4, 0.8]), kind="ry")
        states_broken.append(normalize(unitary_from_hamiltonian(H_break, t) @ psi0))

    cases = [
        ("collapsed",         states_collapsed, H_sym),
        ("symmetry-organised", states_organised, H_sym),
        ("symmetry-breaking",  states_broken,    H_break),
    ]
    results = [compute_symmetry_metrics(s, projectors, H, [Nop], label=lbl) for (lbl, s, H) in cases]
    return results, labels


u1_results, u1_sector_labels = run_u1_trajectories(n=CFG["n_qubits"], T=CFG["T_toy"])
u1_df = pd.DataFrame([{
    "Model":     r["label"],
    "H_G":       r["H_G"],
    "D_G":       r["D_G"],
    "M_G":       r["M_G"],
    "C_G":       r["C_G"],
    "Delta_G":   r["Delta_G"],
    "Psi_G_raw": r["Psi_G_raw"],
    "Psi_G":     r["Psi_G"],
} for r in u1_results])

print("U(1) toy trajectories computed.")
u1_df.round(3)

## 7. Example II: $SU(2)$ total-spin symmetry

The same four qubits, now with total-spin operators $J_\alpha = \tfrac{1}{2}\sum_{i=1}^{4}\sigma_\alpha^{(i)}$ for $\alpha \in \{x, y, z\}$. The Peter–Weyl decomposition is

$$
(\tfrac{1}{2})^{\otimes 4} \cong 2 \cdot (j{=}0) \oplus 3 \cdot (j{=}1) \oplus 1 \cdot (j{=}2),
$$

so $\dim \mathcal L_{SU(2)}(\mathcal H) = 4 + 9 + 1 = 14$. The three trajectories mirror the $U(1)$ construction with the Heisenberg exchange Hamiltonian as the equivariant generator and a local $Z$ plus local $X$ perturbation as the breaking generator.

In [ ]:
def total_spin_operators(n):
    dim = 2**n
    Sx = sum(0.5 * op_on_qubit(X, q, n) for q in range(n))
    Sy = sum(0.5 * op_on_qubit(Y, q, n) for q in range(n))
    Sz = sum(0.5 * op_on_qubit(Z, q, n) for q in range(n))
    S2 = Sx @ Sx + Sy @ Sy + Sz @ Sz
    return Sx, Sy, Sz, S2


def su2_projectors(n):
    Sx, Sy, Sz, S2 = total_spin_operators(n)
    vals, vecs = eigh(S2)
    sectors = {}
    for idx, val in enumerate(vals):
        j = 0.5 * (-1 + np.sqrt(1 + 4*np.real(val)))
        j_round = np.round(2*j) / 2
        sectors.setdefault(j_round, []).append(idx)
    projectors, labels = [], []
    for j in sorted(sectors.keys()):
        V = vecs[:, sectors[j]]
        projectors.append(V @ V.conj().T)
        labels.append(f"$j={j:g}$")
    return projectors, labels, (Sx, Sy, Sz, S2)


def heisenberg_hamiltonian(n, periodic=True):
    H = np.zeros((2**n, 2**n), dtype=complex)
    edges = [(i, i+1) for i in range(n-1)] + ([(n-1, 0)] if periodic and n > 2 else [])
    for i, j in edges:
        H += (two_qubit_op(X, i, X, j, n) + two_qubit_op(Y, i, Y, j, n)
              + two_qubit_op(Z, i, Z, j, n))
    return H


def local_z_disorder(n):
    H = np.zeros((2**n, 2**n), dtype=complex)
    for q, c in enumerate(np.array([0.2, -0.35, 0.55, -0.15])[:n]):
        H += c * op_on_qubit(Z, q, n)
    return H


def run_su2_trajectories(n=4, T=80):
    projectors, labels, (Sx, Sy, Sz, _) = su2_projectors(n)
    H_su2 = heisenberg_hamiltonian(n, periodic=True)
    H_break = H_su2 + 0.8*local_z_disorder(n) + 0.35*local_x_hamiltonian(n)

    # Trajectory 1: collapsed at j=2 sector (the |0000> state has m=2 and j=2).
    psi_jmax = ket0(n)
    states_collapsed = [normalize(unitary_from_hamiltonian(H_su2, t) @ psi_jmax)
                        for t in np.linspace(0, 3.0, T)]

    # Trajectory 2: symmetry-organised over total-spin sectors.
    states_organised = []
    for s, t in enumerate(np.linspace(0, 4.0, T)):
        beta = 0.95 + 0.30*np.sin(2*np.pi*s/T) + 0.12*np.cos(4*np.pi*s/T)
        psi0 = product_rotation_state(n, beta * np.array([0.6, 1.1, 1.4, 0.8]), kind="ry")
        states_organised.append(normalize(unitary_from_hamiltonian(H_su2, t) @ psi0))

    # Trajectory 3: symmetry-breaking Heisenberg + local Z + local X.
    states_broken = []
    for s, t in enumerate(np.linspace(0, 4.0, T)):
        beta = 0.8 + 0.45*np.sin(2*np.pi*s/T + 0.8)
        psi0 = product_rotation_state(n, beta * np.array([1.2, 0.8, 1.0, 1.5]), kind="ry")
        states_broken.append(normalize(unitary_from_hamiltonian(H_break, t) @ psi0))

    cases = [
        ("collapsed",         states_collapsed, H_su2),
        ("symmetry-organised", states_organised, H_su2),
        ("symmetry-breaking",  states_broken,    H_break),
    ]
    # SU(2) compliance requires commuting with Sx, Sy, Sz; S2 alone is not enough.
    generators = [Sx, Sy, Sz]
    results = [compute_symmetry_metrics(s, projectors, H, generators, label=lbl) for (lbl, s, H) in cases]
    return results, labels


su2_results, su2_sector_labels = run_su2_trajectories(n=CFG["n_qubits"], T=CFG["T_toy"])
su2_df = pd.DataFrame([{
    "Model":     r["label"],
    "H_G":       r["H_G"],
    "D_G":       r["D_G"],
    "M_G":       r["M_G"],
    "C_G":       r["C_G"],
    "Delta_G":   r["Delta_G"],
    "Psi_G_raw": r["Psi_G_raw"],
    "Psi_G":     r["Psi_G"],
} for r in su2_results])

print("SU(2) toy trajectories computed.")
su2_df.round(3)

## 8. Example summary

Combine the $U(1)$ and $SU(2)$ tables and save the numerical results to disk.

In [ ]:
u1_out = u1_df.copy();  u1_out.insert(0, "Symmetry", "U(1)")
su2_out = su2_df.copy(); su2_out.insert(0, "Symmetry", "SU(2)")
toy_summary = pd.concat([u1_out, su2_out], ignore_index=True)
toy_summary_round = toy_summary[["Symmetry", "Model", "H_G", "D_G", "M_G",
                                  "C_G", "Delta_G", "Psi_G_raw", "Psi_G"]].round(3)
toy_csv = OUT / "toy_examples_summary.csv"
toy_summary_round.to_csv(toy_csv, index=False)
print(f"Saved {toy_csv}")
toy_summary_round

## 9. Figure 1 — $U(1)$ charge-sector trajectories

Three-panel heatmap of $p_k(t) = \operatorname{Tr}(P_k \rho_t)$ for $k \in \{0,1,2,3,4\}$ along the three $U(1)$ trajectories. Heatmap normalisation is shared across panels so that intensities are directly comparable. The shared colour bar carries the sector probability scale.

In [ ]:
def plot_sector_trajectories(results, sector_labels, title_prefix, filename):
    fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.4), sharey=True,
                             gridspec_kw={"wspace": 0.06}, constrained_layout=False)
    panel_titles = ["(a) collapsed", "(b) symmetry-organised", "(c) symmetry-breaking"]
    ims = []
    for ax, res, panel_title in zip(axes, results, panel_titles):
        im = ax.imshow(res["P_traj"].T, aspect="auto", origin="lower", vmin=0.0, vmax=1.0,
                       cmap="cividis", interpolation="nearest")
        ims.append(im)
        ax.set_title(panel_title, pad=8)
        ax.set_xlabel(r"trajectory step $t$")
        ax.set_yticks(range(len(sector_labels)))
        ax.set_yticklabels(sector_labels)
        ax.tick_params(axis="both", which="major")
    axes[0].set_ylabel(title_prefix)

    fig.subplots_adjust(left=0.08, right=0.88, top=0.88, bottom=0.18)
    cax = fig.add_axes([0.905, 0.18, 0.018, 0.70])
    cbar = fig.colorbar(ims[-1], cax=cax)
    cbar.set_label(r"sector occupation $p_\lambda(t)$")
    cbar.outline.set_linewidth(0.6)
    return save_png(fig, filename)


fig1_path = plot_sector_trajectories(
    u1_results, u1_sector_labels,
    title_prefix=r"$U(1)$ charge sector $k$",
    filename="figure1_u1_sector_trajectories.png")
print(f"Saved {fig1_path}")

## 10. Figure 2 — $SU(2)$ total-spin trajectories

The same three-panel heatmap construction applied to the $SU(2)$ trajectories. The y axis labels total-spin sectors $j \in \{0, 1, 2\}$.

In [ ]:
fig2_path = plot_sector_trajectories(
    su2_results, su2_sector_labels,
    title_prefix=r"$SU(2)$ total-spin sector $j$",
    filename="figure2_su2_sector_trajectories.png")
print(f"Saved {fig2_path}")

## 11. Figure 3 — organisational components and compliance

Two side-by-side panels reporting $H_G$, $D_G$, $M_G$, $C_G$, and $\Psi_G$ for each of the three trajectories of each symmetry group. A single shared legend appears on the right of the figure.

In [ ]:
def plot_component_panels(u1_df, su2_df, filename):
    cols = ["H_G", "D_G", "M_G", "C_G", "Psi_G"]
    pretty = {"H_G": r"$H_G$", "D_G": r"$D_G$", "M_G": r"$M_G$",
              "C_G": r"$C_G$", "Psi_G": r"$\Psi_G$"}

    fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.4), sharex=True, sharey=False,
                             constrained_layout=False)
    titles = [r"$U(1)$ excitation-number symmetry",
              r"$SU(2)$ total-spin symmetry"]
    for ax, df, t in zip(axes, [u1_df, su2_df], titles):
        labels = [wrap_label(x, 22) for x in df["Model"]]
        y = np.arange(len(df))
        height = 0.13
        offsets = (np.arange(len(cols)) - (len(cols)-1)/2) * height
        for off, col, color in zip(offsets, cols, WONG[:len(cols)]):
            ax.barh(y + off, df[col].to_numpy(), height=height, color=color,
                    edgecolor="white", linewidth=0.4, label=pretty[col])
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.invert_yaxis()
        ax.set_xlim(0, 1.06)
        ax.set_xlabel("normalised value")
        ax.set_title(t, pad=10)
        ax.grid(axis="x", linewidth=0.5, alpha=0.25)
        ax.tick_params(axis="y", length=0)
    axes[0].set_ylabel("trajectory")

    handles, labels = axes[1].get_legend_handles_labels()
    fig.subplots_adjust(left=0.13, right=0.88, top=0.9, bottom=0.13, wspace=0.45)
    fig.legend(handles, labels, loc="center left", bbox_to_anchor=(0.895, 0.5),
               frameon=False, handlelength=1.2, labelspacing=1.0)
    return save_png(fig, filename)


fig3_path = plot_component_panels(u1_df, su2_df, "figure3_component_values.png")
print(f"Saved {fig3_path}")

## 12. Figure 4 — $\Psi_G$ across the six toy configurations

Compact horizontal bar plot of $\Psi_G$ for all six toy configurations, with $U(1)$ and $SU(2)$ groups distinguished by colour and individual values labelled at the right of each bar.

In [ ]:
def plot_psi_summary(summary, filename):
    plot_df = summary.copy()
    plot_df["Case"] = plot_df["Symmetry"] + " — " + plot_df["Model"]
    color_map = {"U(1)": WONG[0], "SU(2)": WONG[1]}
    colors = [color_map[s] for s in plot_df["Symmetry"]]

    fig, ax = plt.subplots(figsize=(9.6, 4.6), constrained_layout=False)
    y = np.arange(len(plot_df))
    bars = ax.barh(y, plot_df["Psi_G"].to_numpy(), color=colors,
                   edgecolor="white", linewidth=0.5)
    ax.set_yticks(y)
    ax.set_yticklabels([wrap_label(x, 36) for x in plot_df["Case"]])
    ax.invert_yaxis()
    ax.set_xlim(0, 0.6)
    ax.set_xlabel(r"$\Psi_G$")
    ax.grid(axis="x", linewidth=0.5, alpha=0.25)
    ax.tick_params(axis="y", length=0)

    for yi, val in zip(y, plot_df["Psi_G"].to_numpy()):
        ax.text(val + 0.012, yi, f"{val:.3f}", va="center", fontsize=9.5)

    # Legend for the two symmetry groups.
    from matplotlib.patches import Patch
    leg_handles = [Patch(color=color_map["U(1)"],  label=r"$U(1)$"),
                   Patch(color=color_map["SU(2)"], label=r"$SU(2)$")]
    ax.legend(handles=leg_handles, loc="lower right", frameon=False)

    fig.subplots_adjust(left=0.32, right=0.96, top=0.95, bottom=0.13)
    return save_png(fig, filename)


fig4_path = plot_psi_summary(toy_summary_round, "figure4_psi_summary.png")
print(f"Saved {fig4_path}")

## 13. Trained four-qubit classification task

A reviewer-requested supervised-learning experiment that places the diagnostic $\Psi_G$ alongside train/test performance and gradient statistics. Labels are produced by a fixed $U(1)$-equivariant teacher circuit followed by a non-trivial $U(1)$-invariant readout $O = \sum_k w_k (X_kX_{k{+}1} + Y_kY_{k{+}1})$ on the four-qubit ring. Both the teacher and $O$ commute with the total excitation-number operator $N$, so the task is invariant under the action $\rho \mapsto e^{-i\alpha N}\rho e^{i\alpha N}$. The readout is not a function of $N$ alone, so the student ansatz must act non-trivially within the commutant rather than relying on a trivial charge readout.

Three matched-parameter (8 generators + 1 bias) student ansätze are compared:

- **equivariant:** $XX{+}YY$ exchange and local $Z$ generators only, fully inside the $U(1)$ commutant;
- **hybrid:** mostly equivariant with two controlled local-$X$ symmetry-breaking generators;
- **non-equivariant:** $XX{+}YY$ exchange plus four local-$X$ generators.

Optimisation uses Adam on SPSA gradient estimates (two loss evaluations per epoch regardless of parameter count), 65 epochs, 5 random seeds.

In [ ]:
# ----- Sector-restricted state utilities --------------------------------------
def sector_basis_indices(n, k):
    return [idx for idx in range(2**n) if format(idx, f"0{n}b").count("1") == k]


def random_sector_state(n, k, rng, dicke_mix=0.35):
    '''Random unit vector inside a fixed U(1) charge sector.'''
    dim = 2**n
    inds = sector_basis_indices(n, k)
    v_dicke = np.zeros(dim, dtype=complex)
    v_dicke[inds] = 1.0 / np.sqrt(len(inds))
    coeff = rng.normal(size=len(inds)) + 1j*rng.normal(size=len(inds))
    coeff = coeff / (np.linalg.norm(coeff) + CFG["eps"])
    v_rand = np.zeros(dim, dtype=complex)
    v_rand[inds] = coeff
    psi = dicke_mix*v_dicke + np.sqrt(max(1-dicke_mix**2, 0))*v_rand
    return normalize(psi)


def u1_charge_mixture_state(n, weights, rng):
    psi = np.zeros(2**n, dtype=complex)
    for k, wk in enumerate(weights):
        phase = np.exp(1j*rng.uniform(0, 2*np.pi))
        psi += np.sqrt(max(wk, 0.0)) * phase * random_sector_state(n, k, rng)
    return normalize(psi)


# ----- Ansatz generators (matched parameter counts) ---------------------------
def ring_edges(n):
    return [(i, (i+1) % n) for i in range(n)]


def xy_edge_generator(n, i, j):
    return two_qubit_op(X, i, X, j, n) + two_qubit_op(Y, i, Y, j, n)


def z_generator(n, q):
    return op_on_qubit(Z, q, n)


def normalize_observable(O):
    return O / (fro_norm(O) + CFG["eps"]) * np.sqrt(O.shape[0])


def xy_readout_observable(n=4):
    '''U(1)-invariant XY-edge readout, not a function of total charge alone.'''
    weights = [1.00, -0.75, 0.55, -0.35]
    O = np.zeros((2**n, 2**n), dtype=complex)
    for w, (i, j) in zip(weights, ring_edges(n)):
        O += w * xy_edge_generator(n, i, j)
    return normalize_observable(O)


def ansatz_generators(kind="equivariant", n=4):
    edges = ring_edges(n)
    xy_edges = [0.50 * xy_edge_generator(n, i, j) for i, j in edges]
    z_terms  = [0.65 * z_generator(n, q) for q in range(n)]
    local_x  = [op_on_qubit(X, q, n) for q in range(n)]
    if kind == "equivariant":
        gens = xy_edges + z_terms;                                      label = "trained equivariant"
    elif kind == "hybrid":
        gens = xy_edges + [z_terms[0], 0.65*local_x[1], z_terms[2], 0.65*local_x[3]]; label = "trained hybrid"
    elif kind == "non_equivariant":
        gens = xy_edges + [1.15*G for G in local_x];                    label = "trained non-equivariant"
    else:
        raise ValueError(f"Unknown ansatz kind: {kind}")
    return gens, label


def prepare_generator_cache(generators):
    return [eigh(G) for G in generators]


def unitary_from_cache(cache_item, theta):
    vals, vecs = cache_item
    return (vecs * np.exp(-1j * theta * vals)) @ vecs.conj().T


def apply_ansatz_matrix(Psis, theta, gen_cache):
    out = Psis
    for th, item in zip(theta, gen_cache):
        out = unitary_from_cache(item, th) @ out
    norms = np.linalg.norm(out, axis=0, keepdims=True) + CFG["eps"]
    return out / norms


def expectation_values(Psis, O):
    OPsis = O @ Psis
    return np.real(np.sum(np.conj(Psis) * OPsis, axis=0))


# ----- Teacher-student dataset construction -----------------------------------
def teacher_parameters(n=4):
    return np.array([0.95, -0.70, 0.55, -0.45, 0.80, -0.60, 0.42, -0.35], dtype=float)


def make_classification_dataset(n=4, n_train=60, n_test=40, seed=2026, pool_size=1200):
    '''Balanced U(1)-compatible binary classification dataset.'''
    rng = np.random.default_rng(seed)
    O = xy_readout_observable(n)
    tcache = prepare_generator_cache(ansatz_generators("equivariant", n=n)[0])
    ttheta = teacher_parameters(n)
    states, scores = [], []
    alpha = np.ones(n+1) * 1.4
    for _ in range(pool_size):
        weights = rng.dirichlet(alpha)
        psi = u1_charge_mixture_state(n, weights, rng)
        psi_t = apply_ansatz_matrix(psi.reshape(-1, 1), ttheta, tcache)[:, 0]
        scores.append(float(expectation_values(psi_t.reshape(-1, 1), O)[0]))
        states.append(psi)
    states = np.column_stack(states)
    scores = np.array(scores)
    threshold = float(np.median(scores))
    margin = max(0.04, 0.20*np.std(scores))
    low  = np.where(scores < threshold - margin)[0]
    high = np.where(scores > threshold + margin)[0]
    needed = (n_train + n_test)//2
    if len(low) < needed or len(high) < needed:
        order = np.argsort(scores)
        low  = order[:max(needed, len(order)//3)]
        high = order[-max(needed, len(order)//3):]
    chosen_low  = rng.choice(low,  size=needed, replace=False)
    chosen_high = rng.choice(high, size=needed, replace=False)
    chosen = np.concatenate([chosen_low, chosen_high])
    labels = np.concatenate([np.zeros(needed), np.ones(needed)])
    order = rng.permutation(len(chosen))
    chosen, labels = chosen[order], labels[order]
    Xall = states[:, chosen]
    yall = labels.astype(float)
    return Xall[:, :n_train], yall[:n_train], Xall[:, n_train:n_train+n_test], yall[n_train:n_train+n_test], scores[chosen]


# ----- Training utilities ------------------------------------------------------
def sigmoid(z):
    z = np.clip(z, -35, 35)
    return 1.0 / (1.0 + np.exp(-z))


def predict_probs(Xstates, theta, bias, gen_cache, O):
    Psis = apply_ansatz_matrix(Xstates, theta, gen_cache)
    return sigmoid(CFG["logit_scale"] * expectation_values(Psis, O) + bias)


def bce_loss(y, p, eps=1e-9):
    p = np.clip(p, eps, 1-eps)
    return float(-np.mean(y*np.log(p) + (1-y)*np.log(1-p)))


def loss_for_params(params, Xstates, y, gen_cache, O):
    return bce_loss(y, predict_probs(Xstates, params[:-1], params[-1], gen_cache, O))


def spsa_grad(params, Xstates, y, gen_cache, O, rng, c=None):
    if c is None:
        c = CFG["spsa_c"]
    delta = rng.choice([-1.0, 1.0], size=len(params))
    Lp = loss_for_params(params + c*delta, Xstates, y, gen_cache, O)
    Lm = loss_for_params(params - c*delta, Xstates, y, gen_cache, O)
    return ((Lp - Lm) / (2*c)) * delta


def diagnostic_training_states(Xstates, theta_history, gen_cache, every=5, max_inputs=12):
    Xmat = Xstates[:, :max_inputs]
    states = []
    for theta in theta_history[::every]:
        Psis = apply_ansatz_matrix(Xmat, theta, gen_cache)
        states.extend([Psis[:, i] for i in range(Psis.shape[1])])
    return states


def adam_train(Xtrain, ytrain, Xtest, ytest, kind, seed=0, epochs=None, lr=None):
    if epochs is None: epochs = CFG["epochs"]
    if lr is None:     lr = CFG["lr"]
    rng = np.random.default_rng(seed)
    generators, label = ansatz_generators(kind, n=CFG["n_qubits"])
    gen_cache = prepare_generator_cache(generators)
    O = xy_readout_observable(CFG["n_qubits"])

    params = rng.normal(0.0, CFG["init_sigma"], size=len(generators) + 1)
    m = np.zeros_like(params); v = np.zeros_like(params)
    beta1, beta2 = 0.9, 0.999
    grad_norms, losses = [], []
    theta_history = [params[:-1].copy()]
    for epoch in range(1, epochs+1):
        g = spsa_grad(params, Xtrain, ytrain, gen_cache, O, rng)
        grad_norms.append(float(np.linalg.norm(g)))
        m = beta1*m + (1-beta1)*g
        v = beta2*v + (1-beta2)*(g*g)
        mhat = m/(1-beta1**epoch); vhat = v/(1-beta2**epoch)
        params -= lr*mhat/(np.sqrt(vhat) + 1e-8)
        theta_history.append(params[:-1].copy())
        losses.append(loss_for_params(params, Xtrain, ytrain, gen_cache, O))

    train_probs = predict_probs(Xtrain, params[:-1], params[-1], gen_cache, O)
    test_probs  = predict_probs(Xtest,  params[:-1], params[-1], gen_cache, O)
    train_acc = float(np.mean((train_probs >= 0.5) == ytrain))
    test_acc  = float(np.mean((test_probs  >= 0.5) == ytest))
    final_loss = bce_loss(ytrain, train_probs)

    projectors, _ = excitation_projectors(CFG["n_qubits"])
    Nop = excitation_number_operator(CFG["n_qubits"])
    states_diag = diagnostic_training_states(Xtest, theta_history, gen_cache, every=5, max_inputs=12)
    H_eff = sum(th*G for th, G in zip(params[:-1], generators))
    metrics = compute_symmetry_metrics(states_diag, projectors, H_eff, [Nop], label=label)

    return {
        "Kind": kind, "Model": label, "Seed": seed,
        "Train_acc": train_acc, "Test_acc": test_acc,
        "Final_loss": final_loss,
        "Final_grad_norm": grad_norms[-1],
        "Mean_grad_norm": float(np.mean(grad_norms)),
        "Std_grad_norm":  float(np.std(grad_norms)),
        "Loss_curve":     losses,
        "Grad_norms":     grad_norms,
        **{k: metrics[k] for k in ("H_G", "D_G", "M_G", "C_G", "Delta_G", "Psi_G_raw", "Psi_G")},
    }


def run_trained_experiment(n_seeds=None, epochs=None, dataset_seed=None):
    if n_seeds is None:      n_seeds = CFG["n_seeds"]
    if epochs is None:       epochs  = CFG["epochs"]
    if dataset_seed is None: dataset_seed = CFG["dataset_seed"]
    Xtr, ytr, Xte, yte, scores = make_classification_dataset(
        n=CFG["n_qubits"], n_train=CFG["n_train"], n_test=CFG["n_test"],
        seed=dataset_seed)
    records = []
    for kind in ["equivariant", "hybrid", "non_equivariant"]:
        for seed in range(n_seeds):
            records.append(adam_train(Xtr, ytr, Xte, yte, kind, seed=1000+seed, epochs=epochs))
    return pd.DataFrame(records), (Xtr, ytr, Xte, yte, scores)


print("Trained-task infrastructure ready. Running training (this is the slowest cell)...")
trained_results, trained_data = run_trained_experiment()

trained_summary = trained_results.groupby("Model").agg(
    Train_acc_mean=("Train_acc","mean"), Train_acc_sd=("Train_acc","std"),
    Test_acc_mean =("Test_acc","mean"),  Test_acc_sd =("Test_acc","std"),
    Final_loss_mean=("Final_loss","mean"), Final_loss_sd=("Final_loss","std"),
    Final_grad_mean=("Final_grad_norm","mean"), Final_grad_sd=("Final_grad_norm","std"),
    Mean_grad_mean=("Mean_grad_norm","mean"),
    H_G_mean=("H_G","mean"), H_G_sd=("H_G","std"),
    D_G_mean=("D_G","mean"), D_G_sd=("D_G","std"),
    M_G_mean=("M_G","mean"), M_G_sd=("M_G","std"),
    C_G_mean=("C_G","mean"), C_G_sd=("C_G","std"),
    Psi_G_mean=("Psi_G","mean"), Psi_G_sd=("Psi_G","std"),
).reset_index()

per_seed_csv = OUT / "trained_task_per_seed.csv"
summary_csv  = OUT / "trained_task_summary.csv"
trained_results.drop(columns=["Loss_curve", "Grad_norms"]).to_csv(per_seed_csv, index=False)
trained_summary.to_csv(summary_csv, index=False)
print(f"Saved {per_seed_csv}")
print(f"Saved {summary_csv}")
trained_summary[["Model","Train_acc_mean","Train_acc_sd","Test_acc_mean","Test_acc_sd",
                 "Final_loss_mean","H_G_mean","D_G_mean","M_G_mean","C_G_mean","Psi_G_mean"]].round(3)

## 14. Figures 5, 6, 7 — trained-task results

Three figures summarise the trained-task experiment: train and test accuracy with seed standard deviations (Figure 5), the diagnostic components $H_G$, $D_G$, $M_G$, $C_G$, $\Psi_G$ after training (Figure 6), and gradient-norm statistics across epochs with $\pm$1 SD bands (Figure 7).

A note on Figure 6: the *equivariant* ansatz preserves charge-sector occupation at every iteration by construction, so $H_G$, $D_G$, $M_G$, $C_G$, and $\Psi_G$ are parameter-independent for that ansatz family on a fixed input ensemble and have zero seed variance. This is a property of the diagnostic when applied to an exactly equivariant ansatz, not an implementation artefact.

In [ ]:
def plot_trained_accuracy(summary_df, filename):
    fig, ax = plt.subplots(figsize=(8.6, 4.2), constrained_layout=False)
    y = np.arange(len(summary_df))
    ax.barh(y - 0.18, summary_df["Train_acc_mean"], height=0.32,
            xerr=summary_df["Train_acc_sd"], color=WONG[0],
            edgecolor="white", linewidth=0.5, label="train",
            error_kw=dict(ecolor="#444444", lw=0.8, capsize=2.5))
    ax.barh(y + 0.18, summary_df["Test_acc_mean"], height=0.32,
            xerr=summary_df["Test_acc_sd"], color=WONG[1],
            edgecolor="white", linewidth=0.5, label="test",
            error_kw=dict(ecolor="#444444", lw=0.8, capsize=2.5))
    ax.set_yticks(y)
    ax.set_yticklabels([wrap_label(x, 24) for x in summary_df["Model"]])
    ax.invert_yaxis()
    ax.set_xlim(0, 1.05)
    ax.axvline(0.5, color="#888888", linewidth=0.6, linestyle="--", alpha=0.7, zorder=0)
    ax.set_xlabel(r"accuracy (mean $\pm$ sd over seeds)")
    ax.grid(axis="x", linewidth=0.5, alpha=0.25)
    ax.tick_params(axis="y", length=0)
    ax.legend(loc="lower right", frameon=False)
    fig.subplots_adjust(left=0.27, right=0.97, top=0.95, bottom=0.15)
    return save_png(fig, filename)


def plot_trained_diagnostics(summary_df, filename):
    cols = ["H_G_mean", "D_G_mean", "M_G_mean", "C_G_mean", "Psi_G_mean"]
    err_cols = ["H_G_sd", "D_G_sd", "M_G_sd", "C_G_sd", "Psi_G_sd"]
    pretty = [r"$H_G$", r"$D_G$", r"$M_G$", r"$C_G$", r"$\Psi_G$"]
    fig, ax = plt.subplots(figsize=(10.6, 4.4), constrained_layout=False)
    y = np.arange(len(summary_df))
    height = 0.13
    offsets = (np.arange(len(cols)) - (len(cols)-1)/2) * height
    for off, col, err, lab, color in zip(offsets, cols, err_cols, pretty, WONG[:len(cols)]):
        ax.barh(y + off, summary_df[col].to_numpy(), height=height,
                xerr=summary_df[err].to_numpy(),
                color=color, edgecolor="white", linewidth=0.4, label=lab,
                error_kw=dict(ecolor="#444444", lw=0.8, capsize=2.0))
    ax.set_yticks(y)
    ax.set_yticklabels([wrap_label(x, 24) for x in summary_df["Model"]])
    ax.invert_yaxis()
    ax.set_xlim(0, 1.06)
    ax.set_xlabel(r"normalised value (mean $\pm$ sd over seeds)")
    ax.grid(axis="x", linewidth=0.5, alpha=0.25)
    ax.tick_params(axis="y", length=0)
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
              handlelength=1.4, labelspacing=0.9)
    fig.subplots_adjust(left=0.22, right=0.88, top=0.95, bottom=0.13)
    return save_png(fig, filename)


def plot_gradient_norms(results_df, filename):
    fig, ax = plt.subplots(figsize=(8.6, 4.4), constrained_layout=False)
    model_colors = {m: WONG[i] for i, m in enumerate(results_df["Model"].unique())}
    for model, color in model_colors.items():
        curves = np.array(results_df.loc[results_df["Model"] == model, "Grad_norms"].tolist())
        mean = curves.mean(axis=0)
        sd   = curves.std(axis=0)
        xs = np.arange(1, len(mean)+1)
        ax.plot(xs, mean, label=model, color=color, linewidth=1.8)
        ax.fill_between(xs, np.maximum(mean - sd, 0.0), mean + sd, alpha=0.18, color=color, linewidth=0)
    ax.set_xlabel("training epoch")
    ax.set_ylabel(r"gradient norm $\|\nabla L\|$ (mean $\pm$ sd)")
    ax.set_xlim(1, len(mean))
    ax.grid(linewidth=0.5, alpha=0.25)
    ax.legend(loc="upper right", frameon=False)
    fig.subplots_adjust(left=0.11, right=0.97, top=0.95, bottom=0.13)
    return save_png(fig, filename)


fig5_path = plot_trained_accuracy(trained_summary,    "figure5_trained_task_accuracy.png")
fig6_path = plot_trained_diagnostics(trained_summary, "figure6_trained_task_diagnostics.png")
fig7_path = plot_gradient_norms(trained_results,      "figure7_trained_task_gradient_norms.png")
for p in (fig5_path, fig6_path, fig7_path):
    print(f"Saved {p}")

## 15. Sensitivity sweep and Figure 8

Reviewer 2 requested sensitivity to the component weights $w_H, w_D, w_M$ and the compliance sharpness $\gamma$. The sweep recomputes

$$
\Psi_G(\gamma, \mathbf w) \;=\; \exp(-\gamma \Delta_G)\, \bigl( w_H H_G + w_D D_G + w_M M_G \bigr)
$$

on a fine grid of weight settings on the two-simplex (step 0.05, giving 231 points) for $\gamma \in \{0,1,2,3,5,6,8,10\}$. For each $(\gamma, \mathbf w)$ pair, we record whether the symmetry-organised or equivariant configuration outranks the corresponding symmetry-breaking or non-equivariant configuration. The table reports the percentage of weight settings on which the ranking is preserved, together with the mean and median margin in $\Psi_G$.

In [ ]:
def simplex_weight_grid(step):
    vals = np.arange(0, 1+1e-9, step)
    weights = []
    for wH in vals:
        for wD in vals:
            wM = 1.0 - wH - wD
            if wM >= -1e-9:
                weights.append((float(wH), float(wD), float(max(wM, 0.0))))
    return np.array(weights)


def psi_with_weights(row, weights, gamma):
    raw = weights[:, 0]*row["H_G"] + weights[:, 1]*row["D_G"] + weights[:, 2]*row["M_G"]
    return np.exp(-gamma*row["Delta_G"]) * raw


def sensitivity_pair(df, organised_contains, breaking_contains, gamma_values, step, case_name):
    weights = simplex_weight_grid(step)
    org = df[df["Model"].str.contains(organised_contains, case=False, regex=False)].iloc[0]
    brk = df[df["Model"].str.contains(breaking_contains,  case=False, regex=False)].iloc[0]
    records = []
    for gamma in gamma_values:
        psi_org = psi_with_weights(org, weights, gamma)
        psi_brk = psi_with_weights(brk, weights, gamma)
        records.append({
            "Case": case_name,
            "gamma": gamma,
            "n_weight_settings": len(weights),
            "pct_organised_gt_breaking": 100*np.mean(psi_org > psi_brk),
            "mean_margin":   float(np.mean(psi_org - psi_brk)),
            "median_margin": float(np.median(psi_org - psi_brk)),
        })
    return pd.DataFrame(records)


sens_u1  = sensitivity_pair(u1_df,  "organised", "breaking",
                             CFG["gamma_sweep"], CFG["simplex_step"], "U(1) toy")
sens_su2 = sensitivity_pair(su2_df, "organised", "breaking",
                             CFG["gamma_sweep"], CFG["simplex_step"], "SU(2) toy")

trained_for_sens = trained_summary.rename(columns={
    "H_G_mean":"H_G","D_G_mean":"D_G","M_G_mean":"M_G",
    "C_G_mean":"C_G","Psi_G_mean":"Psi_G"}).copy()
delta_means = trained_results.groupby("Model")["Delta_G"].mean().reset_index()
trained_for_sens = trained_for_sens.merge(delta_means, on="Model", how="left")
sens_trained = sensitivity_pair(trained_for_sens, "equivariant", "non-equivariant",
                                 CFG["gamma_sweep"], CFG["simplex_step"], "trained U(1) task")

sensitivity_summary = pd.concat([sens_u1, sens_su2, sens_trained], ignore_index=True)
sens_csv = OUT / "sensitivity_summary.csv"
sensitivity_summary.to_csv(sens_csv, index=False)
print(f"Saved {sens_csv}")


def plot_sensitivity(summary_df, filename):
    fig, ax = plt.subplots(figsize=(9.0, 5.0), constrained_layout=False)
    markers = ["o", "s", "D"]
    for i, case in enumerate(summary_df["Case"].unique()):
        sub = summary_df[summary_df["Case"] == case]
        ax.plot(sub["gamma"], sub["pct_organised_gt_breaking"],
                marker=markers[i % len(markers)], color=WONG[i],
                markersize=5.5, linewidth=1.7, label=case)
    ax.axvline(CFG["gamma"], color="#888888", linestyle="--", linewidth=0.7, alpha=0.8, zorder=0)
    ax.set_ylim(-2, 104)
    ax.set_xlim(min(CFG["gamma_sweep"])-0.3, max(CFG["gamma_sweep"])+0.3)
    ax.set_xlabel(r"compliance sharpness $\gamma$")
    ax.set_ylabel(r"% weights with organised ranking preserved")
    ax.grid(linewidth=0.5, alpha=0.25)
    ax.legend(loc="lower right", frameon=False)
    fig.subplots_adjust(left=0.10, right=0.97, top=0.95, bottom=0.12)
    return save_png(fig, filename)


fig8_path = plot_sensitivity(sensitivity_summary, "figure8_weight_gamma_sensitivity.png")
print(f"Saved {fig8_path}")
sensitivity_summary.round(2)

## 16. Replication manifest

The final cell writes a JSON manifest of the configuration used for this run, the list of generated artefacts, and the package versions detected at runtime. Reviewers can compare the manifest against the manuscript to confirm that no hyperparameter has drifted between revisions.

In [ ]:
import scipy
manifest = {
    "manuscript":   "Symmetry-Organised Complexity in Quantum Neural Networks (MDPI Symmetry, Revision 1)",
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "environment":  "Google Colab" if RUNNING_IN_COLAB else "local",
    "python":       sys.version.split()[0],
    "platform":     platform.platform(),
    "package_versions": {
        "numpy":      np.__version__,
        "scipy":      scipy.__version__,
        "pandas":     pd.__version__,
        "matplotlib": mpl.__version__,
    },
    "configuration": CFG,
    "output_directory": str(OUT),
    "figures": [
        "figure1_u1_sector_trajectories.png",
        "figure2_su2_sector_trajectories.png",
        "figure3_component_values.png",
        "figure4_psi_summary.png",
        "figure5_trained_task_accuracy.png",
        "figure6_trained_task_diagnostics.png",
        "figure7_trained_task_gradient_norms.png",
        "figure8_weight_gamma_sensitivity.png",
    ],
    "tables": [
        "toy_examples_summary.csv",
        "trained_task_per_seed.csv",
        "trained_task_summary.csv",
        "sensitivity_summary.csv",
    ],
}
manifest_path = OUT / "run_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Saved {manifest_path}")

print("\n" + "="*64)
print("  All outputs written to:")
print(f"    {OUT}")
print("="*64)
print("\nFigures (in manuscript order):")
for fn in manifest["figures"]:
    print(f"  - {fn}")
print("\nTables:")
for fn in manifest["tables"]:
    print(f"  - {fn}")

## 17. Citation

If you reuse any part of this notebook, please cite

> Ugail, H., and Howard, N. Symmetry-Organised Complexity in Quantum Neural Networks, Symmetry, 2026. To Appear

